# Hallucination Detection v2 — Full Pipeline (Colab GPU)

Runs the complete v2 pipeline:
1. **Generate QA pairs** — Claude creates factual questions with known answers
2. **Get model answers** — Pythia-160m answers each question (may hallucinate)
3. **Judge answers** — Claude labels each as `correct / hallucinated / ambiguous`
4. **Extract features** — 5-family 18D feature vector from attention tensors
5. **Train classifier** — Logistic Regression + MLP
6. **Evaluate + ablation** — AUROC, F1, per-family contribution

**Requirements:**
- Colab T4 GPU (free tier)
- Anthropic API key with billing enabled → add as Colab Secret named `ANTHROPIC_API_KEY`

**Outputs:**
- `v2_dataset.jsonl` — labeled dataset
- `v2_detector.pkl` — trained classifier
- `v2_pipeline_results.json` — metrics + ablation
- `v2_feature_distributions.png` — visualisation

**Runtime:** ~15-30 min (scales with `NUM_SAMPLES`)

In [ ]:
# ── 1. Install dependencies ────────────────────────────────────────────────
%pip install -q torch transformers numpy scipy anthropic scikit-learn

import sys, os, torch
print(f"Python {sys.version}")
print(f"PyTorch {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None (CPU mode)'}")

In [ ]:
# ── 2. Load Anthropic API key from Colab Secrets ──────────────────────────
#
# HOW TO ADD YOUR KEY:
#   Left sidebar → 🔑 Secrets → "+ Add new secret"
#   Name:  ANTHROPIC_API_KEY
#   Value: sk-ant-...
#   Toggle "Notebook access" ON
#
# If not on Colab, set the environment variable before running.

if 'google.colab' in sys.modules:
    from google.colab import userdata
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
    print("API key loaded from Colab Secrets.")
else:
    ANTHROPIC_API_KEY = os.environ.get('ANTHROPIC_API_KEY')
    print("API key loaded from environment.")

if not ANTHROPIC_API_KEY:
    raise ValueError("ANTHROPIC_API_KEY not found. See instructions above.")
print(f"Key prefix: {ANTHROPIC_API_KEY[:12]}...")

In [ ]:
# ── 3. Configuration ───────────────────────────────────────────────────────
# Adjust these to control cost and runtime.

NUM_SAMPLES   = 200       # QA pairs to generate  (each uses ~2 API calls)
LOCAL_MODEL   = "EleutherAI/pythia-160m"   # swap for larger model if desired
JUDGE_MODEL   = "claude-sonnet-4-20250514" # Claude model for QA gen + judging
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
SEED          = 42

# Larger model options for better hallucination rate (more interesting data):
#   LOCAL_MODEL = "EleutherAI/pythia-410m"
#   LOCAL_MODEL = "microsoft/phi-2"      # ~2.7B, better hallucinations

print(f"Config:")
print(f"  Samples:      {NUM_SAMPLES}")
print(f"  Local model:  {LOCAL_MODEL}")
print(f"  Judge model:  {JUDGE_MODEL}")
print(f"  Device:       {DEVICE}")

In [ ]:
# ── 4. Clone repo ─────────────────────────────────────────────────────────
if not os.path.exists('Natural-Hallucination-Analysis'):
    !git clone https://github.com/A-Kuo/Natural-Hallucination-Analysis.git

REPO = os.path.abspath('Natural-Hallucination-Analysis')
sys.path.insert(0, REPO)
os.makedirs(f'{REPO}/data', exist_ok=True)

print("Repo:", REPO)

In [ ]:
# ── 5. Step 1/4 — Generate QA pairs via Claude ────────────────────────────
from v2.data_generator import DataGenerator

generator = DataGenerator(
    anthropic_api_key=ANTHROPIC_API_KEY,
    judge_model=JUDGE_MODEL,
)

print(f"Generating {NUM_SAMPLES} QA pairs...")
qa_pairs = generator.generate_qa_pairs(num_pairs=NUM_SAMPLES)
print(f"Generated {len(qa_pairs)} QA pairs")

# Preview
for qa in qa_pairs[:3]:
    print(f"  [{qa.domain}/{qa.difficulty}] {qa.question[:60]}")
    print(f"    → {qa.ground_truth[:80]}")

In [ ]:
# ── 6. Step 2/4 — Get model answers (Pythia on GPU) ───────────────────────
print(f"Loading {LOCAL_MODEL} on {DEVICE}...")
answers = DataGenerator.get_model_answers(
    qa_pairs,
    model_name=LOCAL_MODEL,
    device=DEVICE,
    max_new_tokens=80,
)
print(f"Got {len(answers)} answers.")

# Preview
for qa, ans in zip(qa_pairs[:3], answers[:3]):
    print(f"  Q: {qa.question[:60]}")
    print(f"  A: {ans[:80]}")
    print()

In [ ]:
# ── 7. Step 3/4 — Judge answers with Claude ───────────────────────────────
print("Judging answers...")
labeled_samples = generator.judge_answers(qa_pairs, answers)

# Stats
from collections import Counter
label_counts = Counter(s.label for s in labeled_samples)
total = len(labeled_samples)
print(f"\nDataset: {total} samples")
for label, count in label_counts.items():
    print(f"  {label}: {count} ({count/total*100:.0f}%)")

# Tag with model name and save
for s in labeled_samples:
    s.model_name = LOCAL_MODEL

DATASET_PATH = f'{REPO}/data/v2_dataset.jsonl'
DataGenerator.save(labeled_samples, DATASET_PATH)
print(f"\nSaved dataset: {DATASET_PATH}")

In [ ]:
# ── 8. Step 4/4 — Extract attention features (18D) ────────────────────────
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from v2.feature_engineer import AttentionFeatureEngineer, FeatureConfig
from v2.pipeline import extract_attention_from_model

# Load model for feature extraction (reuse if already in memory)
print(f"Loading {LOCAL_MODEL} for feature extraction...")
tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    LOCAL_MODEL,
    output_attentions=True,
    torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32,
).to(DEVICE).eval()

engineer = AttentionFeatureEngineer()

# Filter out ambiguous
clean_samples = [s for s in labeled_samples if s.label != 'ambiguous']
print(f"Extracting features for {len(clean_samples)} non-ambiguous samples...")

X_list, y_list = [], []
failed = 0

for i, sample in enumerate(clean_samples):
    try:
        text = f"Question: {sample.question}\nAnswer: {sample.model_answer}"
        attentions, context_len = extract_attention_from_model(text, model, tokenizer, DEVICE)
        feats = engineer.extract(attentions, context_len)
        X_list.append(feats)
        y_list.append(1.0 if sample.label == 'hallucinated' else 0.0)
    except Exception as e:
        failed += 1

    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(clean_samples)} processed")

X = np.array(X_list)
y = np.array(y_list)
print(f"\nFeature matrix: {X.shape}  (failed: {failed})")
print(f"Label balance: {int(y.sum())} hallucinated / {int((y==0).sum())} correct")

In [ ]:
# ── 9. Train classifiers ───────────────────────────────────────────────────
from v2.detector import HallucinationDetector
from v2.pipeline import ablation_study

np.random.seed(SEED)
idx = np.random.permutation(len(y))
X, y = X[idx], y[idx]

split = int(0.7 * len(y))
X_train, y_train = X[:split], y[:split]
X_test,  y_test  = X[split:], y[split:]

feature_names = engineer.feature_names

print(f"Training on {split} samples, testing on {len(y)-split}...")

# Logistic Regression
det_lr = HallucinationDetector(classifier_type='logistic', feature_names=feature_names)
det_lr.fit(X_train, y_train)
metrics_lr = det_lr.evaluate(X_test, y_test)

# MLP
det_mlp = HallucinationDetector(classifier_type='mlp', hidden_dim=64, feature_names=feature_names)
det_mlp.fit(X_train, y_train)
metrics_mlp = det_mlp.evaluate(X_test, y_test)

print(f"\nLogistic Regression — AUROC: {metrics_lr.auroc:.4f}  F1: {metrics_lr.f1:.4f}")
print(f"MLP                 — AUROC: {metrics_mlp.auroc:.4f}  F1: {metrics_mlp.f1:.4f}")

In [ ]:
# ── 10. Feature family ablation ────────────────────────────────────────────
ablation_study(X, y, feature_names)

In [ ]:
# ── 11. Feature importance ─────────────────────────────────────────────────
importance = det_lr.feature_importance()
print(f"\nTop feature weights (Logistic Regression):")
for name, weight in list(importance.items())[:12]:
    bar = '█' * int(weight * 25)
    print(f"  {name:<28} {weight:.4f}  {bar}")

In [ ]:
# ── 12. Visualise feature distributions ───────────────────────────────────
import matplotlib.pyplot as plt

family_slices = {
    'Entropy (0-2)':      slice(0, 3),
    'Lookback (3-6)':     slice(3, 7),
    'Frequency (7-10)':   slice(7, 11),
    'Spectral (11-14)':   slice(11, 15),
    'Cross-KL (15-17)':   slice(15, 18),
}

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
fig.suptitle('v2 — Feature Distributions: Correct vs. Hallucinated', fontsize=12)

for ax, (fname, sl) in zip(axes, family_slices.items()):
    correct_mean = X[y == 0][:, sl].mean(axis=1)
    halluc_mean  = X[y == 1][:, sl].mean(axis=1)
    ax.boxplot([correct_mean, halluc_mean],
               labels=['Correct', 'Halluc'],
               patch_artist=True,
               boxprops=dict(alpha=0.7))
    ax.set_title(fname, fontsize=9)

plt.tight_layout()
plt.savefig('v2_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: v2_feature_distributions.png")

In [ ]:
# ── 13. Save detector and results ─────────────────────────────────────────
import json
from datetime import datetime

DETECTOR_PATH = f'{REPO}/data/v2_detector.pkl'
best_det = det_mlp if metrics_mlp.auroc >= metrics_lr.auroc else det_lr
best_det.save(DETECTOR_PATH)
print(f"Saved detector: {DETECTOR_PATH}")

results = {
    "timestamp": datetime.utcnow().isoformat(),
    "model": LOCAL_MODEL,
    "device": DEVICE,
    "num_samples": len(y),
    "train_size": split,
    "test_size": len(y) - split,
    "label_balance": {
        "hallucinated": int(y.sum()),
        "correct": int((y == 0).sum()),
    },
    "logistic_regression": {
        "auroc":     round(metrics_lr.auroc, 4),
        "accuracy":  round(metrics_lr.accuracy, 4),
        "f1":        round(metrics_lr.f1, 4),
        "precision": round(metrics_lr.precision, 4),
        "recall":    round(metrics_lr.recall, 4),
        "fpr":       round(metrics_lr.false_positive_rate, 4),
    },
    "mlp": {
        "auroc":     round(metrics_mlp.auroc, 4),
        "accuracy":  round(metrics_mlp.accuracy, 4),
        "f1":        round(metrics_mlp.f1, 4),
        "precision": round(metrics_mlp.precision, 4),
        "recall":    round(metrics_mlp.recall, 4),
        "fpr":       round(metrics_mlp.false_positive_rate, 4),
    },
    "top_features": {k: round(v, 4) for k, v in list(importance.items())[:10]},
}

RESULTS_PATH = 'v2_pipeline_results.json'
with open(RESULTS_PATH, 'w') as f:
    json.dump(results, f, indent=2)

print(f"Saved results: {RESULTS_PATH}")
print(json.dumps(results["logistic_regression"], indent=2))

In [ ]:
# ── 14. Download outputs ───────────────────────────────────────────────────
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download('v2_pipeline_results.json')
    files.download('v2_feature_distributions.png')
    files.download(DATASET_PATH)     # v2_dataset.jsonl
    files.download(DETECTOR_PATH)    # v2_detector.pkl
    print("Downloads triggered.")
else:
    print("Files saved to:", os.getcwd())

print("\nDone. To update the repo README with real benchmarks, commit the JSON files and update:")
print("  v2/README.md — replace placeholder metrics with values from v2_pipeline_results.json")
print("  README.md    — update the benchmarks table with real AUROC/F1")